In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

:: loading settings :: url = jar:file:/home/coder/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-970c068c-8ca2-49e2-a247-bcc50eb03f02;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickho

In [2]:
#Данные для подключения к табличке постгрес
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD")

#Читаем табличку public.shops
shops_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("user", db_user)
    .option("password", db_password)
    .option("dbtable", "public.shops")
    .option("fetchsize", 1000)
    .option("driver", "org.postgresql.Driver")
    .load()
)


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Загружаем timezone данные из PostgreSQL
timezone_raw_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("user", db_user)
    .option("password", db_password)
    .option("dbtable", "public.shop_timezone")
    .option("driver", "org.postgresql.Driver")
    .load()
)

In [4]:
# 2. Обрабатываем timezone данные
timezone_processed_df = (
    timezone_raw_df
    # Очищаем plant и извлекаем st_id
    .withColumn("plant_clean", F.trim(F.col("plant").cast("string")))
    .filter(F.col("plant_clean").rlike('^[1-9][0-9]*$'))  # только корректные числовые ID
    .withColumn("st_id", F.col("plant_clean").cast("integer"))
    # Очищаем time_zone
    .withColumn("time_zone_clean", 
        F.when(
            (F.trim(F.col("time_zone").cast("string")) == "") | F.col("time_zone").isNull(),
            None
        ).otherwise(F.trim(F.col("time_zone").cast("string")))
    )
    # Дедупликация: для каждого st_id берем не-NULL значение
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("st_id")
                  .orderBy(F.desc_nulls_last("time_zone_clean"))
        )
    )
    .filter(F.col("rn") == 1)
    .select("st_id", "time_zone_clean")
)

In [5]:

# 3. Подготавливаем shops_df
shops_processed_df = shops_df.withColumn("st_id", F.col("st_id").cast("integer"))

In [6]:

# 4. Выполняем join и преобразуем time_zone в tz_code
result_df = (
    shops_processed_df
    .join(timezone_processed_df, on="st_id", how="left")
    .select(
        "st_id",
        "shop_name",
        # Преобразуем time_zone в числовой tz_code
        F.when(
            F.col("time_zone_clean").isNull(), 
            F.lit(3)  # если NULL -> 3
        ).otherwise(
            # Извлекаем число из строки (например, "RUS07" -> 7)
            F.coalesce(
                F.expr("try_cast(TRIM(LEADING '0' FROM regexp_extract(time_zone_clean, '(\\\\d+)', 1)) as int)"),
                F.lit(3)  # если не удалось извлечь число -> 3
            )
        ).alias("tz_code")
    )
    .orderBy("st_id")
)

In [9]:
result_df.show(7)


+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  846|      Lenta|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+
only showing top 7 rows


In [10]:
 spark.stop() #Вырубай нахуй